# Isack — "Hey Isack" Wake Word Training (Clean Rebuild)

This is a consolidated, working version of the openWakeWord custom training notebook,
rebuilt after debugging ~12 separate dependency/compatibility issues on a fresh Colab
runtime (Python 3.13, PyTorch 2.6+, current onnxruntime/onnx versions).

**Before running:** Runtime -> Change runtime type -> T4 GPU.

Run cells top to bottom, in order. Each step includes the fix for a specific bug
we hit the first time through, so this version should run clean end to end.

Expected total time: ~15-25 minutes (mostly the training step).


## Step 1: Install openWakeWord and core dependencies

In [1]:
# Clone openWakeWord and install it WITHOUT its declared dependencies.
# (Its pinned deps include speexdsp-ns, which has no wheel for Python 3.13 and
# breaks the whole pip install otherwise.)
!git clone https://github.com/dscripka/openwakeword
!pip install -e ./openwakeword --no-deps

# Full set of packages the training script (train.py) and its imports actually
# need at runtime. This mirrors the original notebook's install list, minus the
# two packages with no Python 3.13 wheel that aren't actually required:
# speexdsp-ns and tflite-runtime (the .onnx export works fine without either).
!pip install onnxruntime onnxscript scipy tqdm numpy torch torchaudio pyyaml
!pip install mutagen torchinfo torchmetrics speechbrain audiomentations \
    torch-audiomentations acoustics tensorflow_probability pronouncing \
    "datasets==2.14.6" deep-phonemizer webrtcvad

# Download the pre-built feature models (embedding + melspectrogram) that
# ship with openWakeWord releases.
import os
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget -q -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx
!wget -q -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx
print("Step 1 done.")

fatal: destination path 'openwakeword' already exists and is not an empty directory.
Obtaining file:///content/openwakeword
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for openwakeword (pyproject.toml) ... done
  Created wheel for openwakeword: filename=openwakeword-0.6.0-0.editable-py3-none-any.whl size=17481 sha256=d9dfeff1b00fe93c3bf3f583c1b252b41e43d63fd3d97d0a4c84fce3513fd3e7
  Stored in directory: /tmp/pip-ephem-wheel-cache-1gm0xr50/wheels/ab/c1/1e/c1da6751c7c7c120456c5bb1ff850c19b0893c1a0c516fd292
Successfully built openwakeword
  Attempting uninstall: openwakeword
    Found existing installation: openwakeword 0.6.0
    Uninstalling openwakeword-0.6.0:
      Successfully uninstalled openwakeword-0.6.0
Step 1 done.


## Step 2: Get the correct piper-sample-generator commit

The current `main` branch of piper-sample-generator has been restructured and no
longer exposes a top-level `generate_samples` function the way openWakeWord's
training script expects. Commit `a1f84c5` is the version that has it.


In [2]:
!git clone https://github.com/rhasspy/piper-sample-generator /content/piper-sample-generator
!cd /content/piper-sample-generator && git checkout a1f84c5

import os
os.environ["PYTHONPATH"] = "/content/piper-sample-generator:" + os.environ.get("PYTHONPATH", "")

# Verify the right function exists
!grep -n "^def generate_samples" /content/piper-sample-generator/generate_samples.py
print("Step 2 done.")

fatal: destination path '/content/piper-sample-generator' already exists and is not an empty directory.
M	generate_samples.py
M	piper_train/norm_audio/__init__.py
M	piper_train/vits/dataset.py
HEAD is now at a1f84c5 Refactor to allow for usage of generation within Python script, added automatic resampling to 16khz
26:def generate_samples(
Step 2 done.


## Step 3: Install the correct phonemizer + model file

- `piper_phonemize` has no working PyPI wheel for Python 3.13. `piper-phonemize-fix`
  is a community-maintained package that provides working 3.13 wheels and installs
  itself as the same `piper_phonemize` module, so no code changes are needed.
- The model file must be the **v1.0.0** release (`en-us-libritts-high.pt`), matching
  what this commit of piper-sample-generator expects. The v2.0.0 file has a different
  name and will NOT work here.


In [3]:
!pip install piper-phonemize-fix
!apt-get -qq update && apt-get -qq install -y espeak-ng

!wget -q -O /content/piper-sample-generator/models/en-us-libritts-high.pt \
    'https://github.com/rhasspy/piper-sample-generator/releases/download/v1.0.0/en-us-libritts-high.pt'

print("Step 3 done.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Step 3 done.


## Step 4: Patch known PyTorch 2.6+ / torchaudio compatibility breaks

Three real incompatibilities between this older training code and current library
versions. Each is patched at the most robust point possible — modifying the
installed package itself so the fix works regardless of exact wording in whatever
code happens to call it, rather than string-matching against another library's
source (which breaks silently if that library's version changes even slightly).

1. `torchaudio.set_audio_backend` and `torchaudio.info` were both removed from
   current torchaudio. Rather than patching every caller, we add both back
   directly into the installed `torchaudio` package, as thin compatibility
   shims. Anything that calls them (torch_audiomentations or otherwise)
   then just works, unmodified.
2. `torch.load()` now defaults to `weights_only=True`, which rejects the older
   checkpoint files used here. We patch every `torch.load(...)` call found in
   the relevant libraries using a regex that inserts `weights_only=False`,
   rather than hardcoding exact call text.


In [4]:
# Add back torchaudio.set_audio_backend and torchaudio.info as compatibility
# shims, written directly into the installed torchaudio package on disk (not
# just the current Python session) so the fix also applies inside the
# subprocess that train.py runs in.
import torchaudio, os

torchaudio_init_path = os.path.join(os.path.dirname(torchaudio.__file__), "__init__.py")

with open(torchaudio_init_path, "r") as f:
    ta_content = f.read()

shim_code = '''

# --- compatibility shim added for older training code (openwakeword/isack) ---
# NOTE: this code executes AS PART OF torchaudio's own module body (it's
# appended to torchaudio/__init__.py), so it must check/define things via
# globals(), not via a self-referencing "torchaudio.x" name -- the module
# does not see itself under that name while its own file is executing.
if "set_audio_backend" not in globals():
    def set_audio_backend(backend):
        pass

if "info" not in globals():
    import soundfile as _sf
    class _AudioMetaDataShim:
        def __init__(self, num_frames, sample_rate):
            self.num_frames = num_frames
            self.sample_rate = sample_rate
    def info(filepath, **kwargs):
        _info = _sf.info(filepath)
        return _AudioMetaDataShim(_info.frames, _info.samplerate)
# --- end compatibility shim ---
'''

marker = "# --- compatibility shim added for older training code (openwakeword/isack) ---"
if marker not in ta_content:
    with open(torchaudio_init_path, "a") as f:
        f.write(shim_code)
    print(f"Compatibility shim appended to {torchaudio_init_path}")
else:
    print("Shim already present, skipping.")

# Verify it actually works when torchaudio is freshly imported (simulating
# what the train.py subprocess will see). Written to a temp file rather than
# an inline -c string to avoid shell quote-escaping issues entirely.
verify_script = '''
import torchaudio
torchaudio.set_audio_backend("test")
print("set_audio_backend OK, info() defined:", hasattr(torchaudio, "info"))
'''
with open("/tmp/verify_torchaudio_shim.py", "w") as f:
    f.write(verify_script)

!python3 /tmp/verify_torchaudio_shim.py

print("Step 4a done.")

Shim already present, skipping.
set_audio_backend OK, info() defined: True
Step 4a done.


In [5]:
# Patch every torch.load(...) call found in the relevant libraries so
# PyTorch 2.6+'s weights_only=True default doesn't block loading these
# older checkpoint files. Uses a regex rather than hardcoded call text, so
# it catches any call shape (any arguments), not just the specific ones we
# happened to encounter.
import glob, re

files_to_check = glob.glob("/usr/local/lib/python3.13/dist-packages/dp/**/*.py", recursive=True)
files_to_check += glob.glob("/content/piper-sample-generator/**/*.py", recursive=True)
files_to_check += glob.glob("/content/openwakeword/**/*.py", recursive=True)

# Matches torch.load(<anything>) allowing one level of nested parens,
# e.g. torch.load(os.path.join(a, b)) as well as simple calls.
load_pattern = re.compile(r"torch\.load\(((?:[^()]|\([^()]*\))*)\)")

def patch_call(match):
    args = match.group(1)
    if "weights_only" in args:
        return match.group(0)
    args = args.rstrip()
    sep = ", " if args else ""
    return f"torch.load({args}{sep}weights_only=False)"

patched = []
for file_path in files_to_check:
    with open(file_path, "r") as f:
        content = f.read()
    if "torch.load(" not in content:
        continue
    new_content = load_pattern.sub(patch_call, content)
    if new_content != content:
        with open(file_path, "w") as f:
            f.write(new_content)
        patched.append(file_path)

print("Patched files:", patched if patched else "(none needed patching)")
print("Step 4b done.")

Patched files: (none needed patching)
Step 4b done.


In [6]:
print("Step 4 done — torchaudio and torch.load compatibility fixes applied.")

Step 4 done — torchaudio and torch.load compatibility fixes applied.


## Step 5: Download training/validation data

In [7]:
import os, numpy as np, torch, scipy, datasets
from pathlib import Path
from tqdm import tqdm

# Room impulse responses (MIT)
output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
for row in tqdm(rir_dataset, desc="RIRs"):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# Background noise (AudioSet sample)
if not os.path.exists("audioset"):
    os.mkdir("audioset")
fname = "bal_train09.tar"
out_dir = f"audioset/{fname}"
link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
!wget -q -O {out_dir} {link}
!cd audioset && tar -xf bal_train09.tar

output_dir = "./audioset_16k"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset, desc="AudioSet"):
    name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# Free Music Archive (background music)
output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))
n_hours = 1
for i in tqdm(range(n_hours*3600//30), desc="FMA"):
    row = next(fma_dataset)
    name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
    i += 1
    if i == n_hours*3600//30:
        break

print("Step 5 done.")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

RIRs: 270it [01:33,  2.88it/s]


tar: This does not look like a tar archive
tar: Exiting with failure status due to previous errors


AudioSet: 0it [00:00, ?it/s]
FMA:  99%|█████████▉| 119/120 [00:47<00:00,  2.49it/s]

Step 5 done.


In [8]:
# Pre-computed negative features + validation set (large downloads, ~16GB + 176MB)
!wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy
print("Feature downloads done.")

Feature downloads done.


## Step 6: Configure training

In [9]:
import yaml

config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)

config["target_phrase"] = ["hey isack"]
config["model_name"] = config["target_phrase"][0].replace(" ", "_")
config["n_samples"] = 1000
config["n_samples_val"] = 1000
config["steps"] = 10000
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25

config["background_paths"] = ['./audioset_16k', './fma']
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    yaml.dump(config, file)

print("Config saved. Target phrase:", config["target_phrase"])

Config saved. Target phrase: ['hey isack']


## Step 7: Generate training clips

Note: the first run will show a warning that 'isack' isn't in the standard
pronunciation dictionary and will use DeepPhonemizer to predict the phonemes
automatically ([IH][S][AH][K]). This is expected and fine.


In [10]:
!pip install mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 \
    speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.11.0 \
    acoustics==0.2.6 tensorflow_probability==0.16.0 pronouncing==0.2.0 \
    "datasets==2.14.6" deep-phonemizer==0.0.19 webrtcvad

In [16]:
!pip install espeak-phonemizer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 75.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for espeak-phonemizer: filename=espeak_phonemizer-1.3.1-py3-none-any.whl size=19869 sha256=cf3cb4c552974c507d3f4c7283c9b5babe66d2a57001c4eca96df3b90e11839d
  Stored in directory: /root/.cache/pip/wheels/1b/2f/af/3094b23040149ae761f2dea9e264e5a2e0695cfae2a2d52cdf
Successfully built espeak-phonemizer


In [17]:
import sys
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips

/usr/local/lib/python3.13/dist-packages/pronouncing/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream
INFO:root:##################################################
Generating positive clips for training
##################################################
DEBUG:generate_samples:Loading /content/piper-sample-generator/models/en-us-libritts-high.pt
INFO:generate_samples:Successfully loaded the model
DEBUG:generate_samples:CUDA available, using GPU
/content/piper-sample-generator/generate_samples.py:106: UserWarning: "kaiser_window" resampling method name is being deprecated and replaced by "sinc_interp_kaiser" in the next release. The default behavior remains unchanged.
  resampler = torchaudio.transforms.Resample(
INFO:root:###################

## Step 8: Augment clips and compute features

Known bug: the "already exists, skipping" check in this step only checks for
the *training* features file, not the *test* one. If a previous partial run
left a training features file behind without its matching test file, this
step will wrongly skip and Step 9 will fail with a FileNotFoundError. The
cell below clears any partial output first so this always runs for real.


In [18]:
model_name = config["model_name"]
feature_dir = f"/content/my_custom_model/{model_name}"
import os
for fname in ["positive_features_train.npy", "positive_features_test.npy",
              "negative_features_train.npy", "negative_features_test.npy"]:
    fpath = os.path.join(feature_dir, fname)
    if os.path.exists(fpath):
        os.remove(fpath)

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

/usr/local/lib/python3.13/dist-packages/pronouncing/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream
INFO:root:##################################################
Computing openwakeword features for generated samples
##################################################
/usr/local/lib/python3.13/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:153: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch_audiomentations/core/transforms_interface.py:77: FutureWarning: Transforms now expect an `output_type` argument that currently defaults to 'tensor', will default

## Step 9: Train the model

Installing `onnxscript` here, before training starts, rather than after — the
export step at the end of training needs it, and if it's missing, the training
script fails only at that final export step. Re-running `--train_model` at that
point does **not** resume from the finished checkpoint, it retrains from
scratch, wasting the entire ~10-15 minutes over again. Installing it up front
avoids that.

This is the long step (~10-15 min on a T4). It runs 3 training sequences,
progressively increasing the weight on negative examples to reduce false
positives, then exports the result to `.onnx`.

**Note:** this notebook does NOT attempt the `.tflite` export (an optional step
in the original notebook). That conversion depends on `onnx_tf` /
`tensorflow-addons`, which have no working install path on current Python/TF
versions. The `.onnx` file alone is fully sufficient to run openWakeWord.


In [20]:
!ls -la /content/my_custom_model/

total 224
drwxr-xr-x 3 root root   4096 Sep  7 22:22 .
drwxr-xr-x 1 root root   4096 Sep  7 22:04 ..
drwxr-xr-x 6 root root   4096 Sep  7 22:10 hey_isack
-rw-r--r-- 1 root root  13814 Sep  7 22:22 hey_isack.onnx
-rw-r--r-- 1 root root 200704 Sep  7 22:22 hey_isack.onnx.data


In [19]:
!pip install onnxscript
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

/usr/local/lib/python3.13/dist-packages/pronouncing/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream
INFO:root:##################################################
Starting training sequence 1...
##################################################
Training: 100% 9999/10000 [05:28<00:00, 30.44it/s]
INFO:root:##################################################
Starting training sequence 2...
##################################################
INFO:root:Increasing weight on negative examples to reduce false positives...
Training: 100% 999/1000.0 [02:32<00:00,  6.55it/s]
INFO:root:##################################################
Starting training sequence 3...
##################################################
INFO:root:Increasing weight on nega

In [22]:
!ls -la /content/my_custom_model/hey_isack.onnx.data

-rw-r--r-- 1 root root 200704 Sep  7 22:22 /content/my_custom_model/hey_isack.onnx.data


In [23]:
from google.colab import files
files.download('/content/my_custom_model/hey_isack.onnx.data')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 10: If the export step still fails

Only run this cell if Step 9 finished training (you'll see a line like
`Final Model Accuracy: ...`) but then errored during ONNX export. This
re-runs the same command, which will retrain from scratch (~10-15 min again)
before reaching export — it exists as a fallback, not something you should
need if Step 9 succeeded cleanly.


In [14]:
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

/usr/local/lib/python3.13/dist-packages/pronouncing/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream
Traceback (most recent call last):
  File "/content/openwakeword/openwakeword/train.py", line 646, in <module>
    from generate_samples import generate_samples
  File "/content/piper-sample-generator/generate_samples.py", line 18, in <module>
    from espeak_phonemizer import Phonemizer
ModuleNotFoundError: No module named 'espeak_phonemizer'


## Step 11: Verify and download the trained model

In [15]:
model_name = config["model_name"]
!ls -la /content/my_custom_model/

from google.colab import files
files.download(f'/content/my_custom_model/{model_name}.onnx')
print(f"Downloading {model_name}.onnx -- move it into your isack project folder once it lands.")

ls: cannot access '/content/my_custom_model/': No such file or directory


FileNotFoundError: Cannot find file: /content/my_custom_model/hey_isack.onnx

## Training a second wake word (e.g. "Hey Irin")

This notebook already parameterizes everything off `config["target_phrase"]`
(the model name, output filenames, and feature directory are all derived from
it in Step 6), so you don't need a second copy of the notebook. To train an
additional wake word once Step 11 has finished for "hey isack":

1. Go back to **Step 6** and change the config:
   ```python
   config["target_phrase"] = ["hey irin"]
   ```
2. Re-run **Step 6 through Step 11** (you can skip Steps 1-5 — the installed
   packages, piper model, RIRs/background noise, and precomputed negative
   features are all still loaded in the runtime and are reused as-is).
3. This produces `hey_irin.onnx` (and `hey_irin.onnx.data`), downloaded the
   same way as before.
4. Drop both files into the local `isack` project folder alongside
   `hey_isack.onnx`. `detect_isack.py` and `isack_listener.py` already look
   for a `hey_irin.onnx` file and will pick it up automatically — no code
   changes needed.

## Next steps (outside this notebook)

1. Move `hey_isack.onnx` into your local `isack` project folder.
2. `pip install openwakeword` (the pip-installable runtime package, separate
   from the training repo cloned here) in your local venv.
3. Write a detection script that loads this custom model and listens via your
   Mac's microphone in real time.
4. Commit the `.onnx` file and this notebook to your GitHub repo.
